In [ ]:
# Practice Problems Day 11

### Author: Joanna Bieri (solutions)

## Introduction to Machine Learning

#### University of Redlands - DATA 301
#### Prof: Joanna Bieri [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
#### Day 11 Practice - SOLUTIONS
#### [Class Website](https://joannabieri.com/machine_learning.html)

---

**These are practice problems. They are not graded and you do not turn them in.** They are here so you can practice today's code before the weekly homework. Weekly Homework 6 assumes you have done them. Plan on about 45 minutes.

**These are my solutions.** Try every problem yourself first, then come here to check your work. There is more than one right way to write the code, and your words will be different from mine, that is fine! If your numbers do not match, look for the difference before you look at my code.

**Reading:** Geron Appendix A (Autodiff), and chapter 11, **Glorot and He Initialization** and **Better Activation Functions**. (Remember, PyTorch calls these Xavier and Kaiming.)

GOALS:

1. Do backpropagation by hand on a tiny network.
2. Write a deep network from start to finish, with Kaiming initialization.
3. Explain vanishing gradients in words.

**How much of this should you write yourself?** Problem 1 is by hand, and the check cell is given, you just run it. Problem 2 lists the steps. Problem 3 is words only. The **Optional exploration** at the end is extra, only if you are curious.

**Save your work.** Commit and push when you are done, so you have your code for the weekly homework and the exam:

```bash
git add .
git commit -m "Day 11 practice"
git push
```

---

In [ ]:
# RUN THIS. The California housing data from the notes, split, scaled, and turned into tensors.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

housing = pd.read_csv("data/california_housing.csv")
X = housing.drop(columns=["MedHouseVal"])
y = housing["MedHouseVal"]
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.25, random_state=42)

preprocessing = Pipeline([("scale", StandardScaler())])      # the preprocessing pipeline, fit on the training rows only
X_train_scaled = preprocessing.fit_transform(X_train)
X_valid_scaled = preprocessing.transform(X_valid)

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
X_valid_t = torch.tensor(X_valid_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train.values, dtype=torch.float32).reshape(-1, 1)
y_valid_t = torch.tensor(y_valid.values, dtype=torch.float32).reshape(-1, 1)
train_data = TensorDataset(X_train_t, y_train_t)               # pairs the features with the labels for the DataLoader

print("training rows:", X_train_t.shape[0], "   validation rows:", X_valid_t.shape[0])

## Problem 1: Backpropagation by hand

This is the tiny network from the notes, with different numbers: one input, one hidden neuron with a ReLU, one output neuron.

$$x = 1, \quad y = 2, \quad w_1 = 2, \quad b_1 = -1, \quad w_2 = 0.5, \quad b_2 = 1$$

**1a.** By hand, do the forward pass: find $z$, $h$, $\hat{y}$ and the loss $L = (\hat{y} - y)^2$. Then do the backward pass and find the four derivatives $\partial L / \partial w_1$, $\partial L / \partial b_1$, $\partial L / \partial w_2$ and $\partial L / \partial b_2$. Use the pictures in the notes as your guide.

**1b.** Run the cell below to check your answers with PyTorch.

**1c.** In the cell below, change $b_1$ to $-3$ and run it again. What happens to the derivatives for $w_1$ and $b_1$? Explain why, using what you know about the ReLU.

In [ ]:
# RUN THIS. Check your answers to 1a with PyTorch.
x = torch.tensor(1.0)                          # the input
y_true = torch.tensor(2.0)                     # the right answer

w1 = torch.tensor(2.0, requires_grad=True)     # requires_grad=True tells PyTorch to keep track of the derivatives
b1 = torch.tensor(-1.0, requires_grad=True)    # for 1c, change this to -3.0
w2 = torch.tensor(0.5, requires_grad=True)
b2 = torch.tensor(1.0, requires_grad=True)

# The forward pass
z = w1 * x + b1                                # the weighted sum of the hidden neuron
h = torch.relu(z)                              # the activation
y_hat = w2 * h + b2                            # the output neuron
loss = (y_hat - y_true) ** 2                   # the squared error

# The backward pass
loss.backward()

print("z:", z.item(), "  h:", h.item(), "  y_hat:", y_hat.item(), "  loss:", loss.item())
print("dL/dw1 =", w1.grad.item(), "  dL/db1 =", b1.grad.item(), "  dL/dw2 =", w2.grad.item(), "  dL/db2 =", b2.grad.item())

In [ ]:
# 1c: the same cell with b1 = -3.0
x = torch.tensor(1.0)                          # the input
y_true = torch.tensor(2.0)                     # the right answer

w1 = torch.tensor(2.0, requires_grad=True)     # requires_grad=True tells PyTorch to keep track of the derivatives
b1 = torch.tensor(-3.0, requires_grad=True)    # changed for 1c
w2 = torch.tensor(0.5, requires_grad=True)
b2 = torch.tensor(1.0, requires_grad=True)

# The forward pass
z = w1 * x + b1                                # the weighted sum of the hidden neuron
h = torch.relu(z)                              # the activation
y_hat = w2 * h + b2                            # the output neuron
loss = (y_hat - y_true) ** 2                   # the squared error

# The backward pass
loss.backward()

print("z:", z.item(), "  h:", h.item(), "  y_hat:", y_hat.item(), "  loss:", loss.item())
print("dL/dw1 =", w1.grad.item(), "  dL/db1 =", b1.grad.item(), "  dL/dw2 =", w2.grad.item(), "  dL/db2 =", b2.grad.item())

**1a.** The forward pass, left to right:

- $z = w_1 x + b_1 = 2 \cdot 1 - 1 = 1$
- $h = \text{ReLU}(1) = 1$
- $\hat{y} = w_2 h + b_2 = 0.5 \cdot 1 + 1 = 1.5$
- $L = (1.5 - 2)^2 = 0.25$

The backward pass, right to left, multiplying by one more derivative at each step:

- $\partial L / \partial \hat{y} = 2(\hat{y} - y) = 2(1.5 - 2) = -1$
- $\partial L / \partial w_2 = -1 \cdot h = -1$ and $\partial L / \partial b_2 = -1$
- $\partial L / \partial h = -1 \cdot w_2 = -0.5$
- Through the ReLU: $z$ is positive, so we multiply by 1 and $\partial L / \partial z = -0.5$
- $\partial L / \partial w_1 = -0.5 \cdot x = -0.5$ and $\partial L / \partial b_1 = -0.5$

**1b.** PyTorch gets the same thing: loss 0.25, and the derivatives $-0.5$, $-0.5$, $-1.0$, $-1.0$.

**1c.** With $b_1 = -3$ we get $z = 2 - 3 = -1$. That is negative, so the ReLU gives $h = 0$ and the loss is $(1 - 2)^2 = 1$.

The derivatives for $w_1$ and $b_1$ are both **0**. The derivative of ReLU is 0 when $z$ is negative, so going back through the ReLU multiplies the gradient by 0 and nothing gets back to the hidden neuron. $\partial L / \partial w_2$ is 0 too, because it is $\partial L / \partial \hat{y}$ times $h$, and $h$ is 0. Only $b_2$ gets a gradient ($-2$).

If this happened for every row of data, the hidden neuron would never change. That is a dead ReLU!

NOTE - PyTorch prints `-0.0` for $\partial L / \partial w_2$. That is just 0.

---

## Problem 2: A deep network from start to finish (steps outlined)

**What to copy, what to write:** Use the "What this looks like in real life" cell at the end of the notes as your guide, but type it out yourself. This is the pattern you need for the weekly homework.

**2a.** Write one cell that builds a network with **4 hidden layers** of 40 neurons each, with ReLU after each one, gives it Kaiming initialization, and trains it with SGD, a learning rate of 0.001, for 20 epochs. Print the validation RMSE after the first epoch and then every 5 epochs.

**2b.** Copy your cell into the next cell and delete the Kaiming initialization loop, so the network uses PyTorch's default starting weights. Run it.

**2c.** Compare the validation RMSE after 1 epoch and after 20 epochs. Which starting weights got going faster? Use what you learned in the notes to explain why.

In [ ]:
# 2a: 4 hidden layers, ReLU, Kaiming initialization
torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(8, 40),                         # 8 features in, 40 neurons
    nn.ReLU(),
    nn.Linear(40, 40),
    nn.ReLU(),
    nn.Linear(40, 40),
    nn.ReLU(),
    nn.Linear(40, 40),
    nn.ReLU(),
    nn.Linear(40, 1))                         # 1 prediction out

# the starting weights
for layer in model:
    if isinstance(layer, nn.Linear):          # only the Linear layers have weights
        nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
        nn.init.zeros_(layer.bias)

# the batches, the loss and the optimizer
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

# the training loop
n_epochs = 20
for epoch in range(n_epochs):
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()                 # clear the old gradients
        y_pred = model(X_batch)               # forward pass
        loss = loss_fn(y_pred, y_batch)       # how far off are we?
        loss.backward()                       # backward pass, this is backpropagation
        optimizer.step()                      # one step of gradient descent

    # after the first epoch and every 5 epochs, check the validation RMSE
    if epoch == 0 or (epoch + 1) % 5 == 0:
        with torch.no_grad():
            rmse = torch.sqrt(loss_fn(model(X_valid_t), y_valid_t)).item()
        print("epoch", epoch + 1, "  valid RMSE:", round(rmse, 4))

In [ ]:
# 2b: the same network with PyTorch's default starting weights (the Kaiming loop is gone)
torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(8, 40),                         # 8 features in, 40 neurons
    nn.ReLU(),
    nn.Linear(40, 40),
    nn.ReLU(),
    nn.Linear(40, 40),
    nn.ReLU(),
    nn.Linear(40, 40),
    nn.ReLU(),
    nn.Linear(40, 1))                         # 1 prediction out

# the batches, the loss and the optimizer
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

# the training loop
n_epochs = 20
for epoch in range(n_epochs):
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()                 # clear the old gradients
        y_pred = model(X_batch)               # forward pass
        loss = loss_fn(y_pred, y_batch)       # how far off are we?
        loss.backward()                       # backward pass, this is backpropagation
        optimizer.step()                      # one step of gradient descent

    # after the first epoch and every 5 epochs, check the validation RMSE
    if epoch == 0 or (epoch + 1) % 5 == 0:
        with torch.no_grad():
            rmse = torch.sqrt(loss_fn(model(X_valid_t), y_valid_t)).item()
        print("epoch", epoch + 1, "  valid RMSE:", round(rmse, 4))

**2a and 2b.** Here are my numbers:

| starting weights | after 1 epoch | after 20 epochs |
|---|---|---|
| Kaiming | 0.8788 | 0.6388 |
| default | 1.1403 | 0.6565 |

**2c.** Kaiming initialization got going much faster. After one epoch it was already at 0.8788, while the default network was at 1.1403, which is about what we get by always predicting the average (1.1503). After 20 epochs Kaiming is still a little ahead, 0.6388 against 0.6565.

Why? PyTorch's default starting weights are on the small side for a deep ReLU network, so the signal shrinks as it goes forward through the layers, and the gradients shrink as they go back. The first epochs are spent barely moving. With Kaiming initialization the signal and the gradients are a good size from the very first step.

NOTE - With 4 hidden layers the default network does get going after a few epochs. In the notes, with 10 hidden layers and the same learning rate, it never did. The deeper the network, the more the starting weights matter.

---

## Problem 3: In words

**3a.** Explain **vanishing gradients** to a friend in DATA 101 who knows what a derivative is but has never seen a neural network. Three or four sentences, no code.

**3b.** A classmate builds a network with 20 hidden layers and sigmoid activations to predict house values. After 30 epochs the validation RMSE is exactly what you get by always predicting the average. Name two things they should change, and say why each one helps.

**3a.** Your answer will be different, here is one way to say it:

A neural network learns by using the derivative of its error to decide how to change each of its numbers. In a network with many layers, the derivative for the first layer is a product of one number for every layer after it. If those numbers are all smaller than 1 (with the sigmoid they are 0.25 or less), the product gets tiny. So the first layers get told to change by almost nothing, and they never learn.

The key ideas: the gradient is a product of many numbers, the product shrinks, so the early layers do not change.

**3b.** Two good changes:

1. **Use ReLU instead of the sigmoid** (or Leaky ReLU or ELU). The derivative of the sigmoid is at most 0.25, so going back through 20 sigmoid layers shrinks the gradient to nothing. The derivative of ReLU is 1 for positive numbers, so it does not shrink.
2. **Use Kaiming initialization** for the starting weights, so the signal going forward and the gradients going back stay a good size through all 20 layers. They might need a smaller learning rate after that.

Using fewer layers is also a fair answer!

---

## Optional exploration

Only if you are curious. Nothing here is needed for the weekly homework.

**A.** In your 2a cell, change every `nn.ReLU()` to `nn.LeakyReLU()`. Is it any different?

**B.** In your 2a cell, replace the Kaiming line with `nn.init.normal_(layer.weight, std=1.0)`, starting weights that are much too big. What happens, and why?

In [ ]:
# Optional A: Leaky ReLU instead of ReLU
torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(8, 40),                         # 8 features in, 40 neurons
    nn.LeakyReLU(),
    nn.Linear(40, 40),
    nn.LeakyReLU(),
    nn.Linear(40, 40),
    nn.LeakyReLU(),
    nn.Linear(40, 40),
    nn.LeakyReLU(),
    nn.Linear(40, 1))                         # 1 prediction out

# the starting weights
for layer in model:
    if isinstance(layer, nn.Linear):          # only the Linear layers have weights
        nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
        nn.init.zeros_(layer.bias)

# the batches, the loss and the optimizer
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

# the training loop
n_epochs = 20
for epoch in range(n_epochs):
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()                 # clear the old gradients
        y_pred = model(X_batch)               # forward pass
        loss = loss_fn(y_pred, y_batch)       # how far off are we?
        loss.backward()                       # backward pass, this is backpropagation
        optimizer.step()                      # one step of gradient descent

    # after the first epoch and every 5 epochs, check the validation RMSE
    if epoch == 0 or (epoch + 1) % 5 == 0:
        with torch.no_grad():
            rmse = torch.sqrt(loss_fn(model(X_valid_t), y_valid_t)).item()
        print("epoch", epoch + 1, "  valid RMSE:", round(rmse, 4))

In [ ]:
# Optional B: starting weights that are much too big
torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(8, 40),                         # 8 features in, 40 neurons
    nn.ReLU(),
    nn.Linear(40, 40),
    nn.ReLU(),
    nn.Linear(40, 40),
    nn.ReLU(),
    nn.Linear(40, 40),
    nn.ReLU(),
    nn.Linear(40, 1))                         # 1 prediction out

# the starting weights
for layer in model:
    if isinstance(layer, nn.Linear):          # only the Linear layers have weights
        nn.init.normal_(layer.weight, std=1.0)
        nn.init.zeros_(layer.bias)

# the batches, the loss and the optimizer
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.001)

# the training loop
n_epochs = 20
for epoch in range(n_epochs):
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()                 # clear the old gradients
        y_pred = model(X_batch)               # forward pass
        loss = loss_fn(y_pred, y_batch)       # how far off are we?
        loss.backward()                       # backward pass, this is backpropagation
        optimizer.step()                      # one step of gradient descent

    # after the first epoch and every 5 epochs, check the validation RMSE
    if epoch == 0 or (epoch + 1) % 5 == 0:
        with torch.no_grad():
            rmse = torch.sqrt(loss_fn(model(X_valid_t), y_valid_t)).item()
        print("epoch", epoch + 1, "  valid RMSE:", round(rmse, 4))

**A.** Almost no difference. With Kaiming initialization, Leaky ReLU ends at 0.6408 and ReLU at 0.6388.

**B.** The loss is `nan` right away. With a standard deviation of 1 the starting weights are much too big, so the signal gets bigger at every layer and the gradients explode. This is the "too big" case from the notes.